<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
!pip3 install beautifulsoup4
!pip3 install requests

In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings]
    return out[0] if out else None


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
response = requests.get(static_url, headers=headers, timeout=40)
response.raise_for_status()
print('HTTP status:', response.status_code, '| HTML characters:', len(response.text))

HTTP status: 200 | HTML characters: 3289905


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
soup = BeautifulSoup(response.text, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
print(soup.title.get_text(' ', strip=True) if soup.title else 'No HTML title found')

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
html_tables = soup.find_all('table')
print('Tables found:', len(html_tables))

Tables found: 29


Starting from the third table is our target table contains the actual launch records.


In [9]:
launch_tables = soup.select('table.wikitable.plainrowheaders')
if not launch_tables:
    launch_tables = [t for t in html_tables if t.find('th', string=lambda s: bool(s and 'Flight No.' in s))]
if not launch_tables:
    raise RuntimeError('No launch tables found: Wikipedia page structure may have changed.')
first_launch_table = launch_tables[0]
print(first_launch_table.prettify()[:1200])

<table class="wikitable plainrowheaders collapsible" id="mwmQ" style="width: 100%;">
 <tbody id="mwmg">
  <tr id="mwmw">
   <th id="mwnA" scope="col">
    Flight No.
   </th>
   <th id="mwnQ" scope="col">
    Date and
    <br id="mwng"/>
    time (
    <a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwnw" rel="mw:WikiLink" title="Coordinated Universal Time">
     UTC
    </a>
    )
   </th>
   <th id="mwoA" scope="col">
    <a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwoQ" rel="mw:WikiLink" title="List of Falcon 9 first-stage boosters">
     Version,
     <br id="mwog"/>
     Booster
    </a>
    <sup about="#mwt70" class="mw-ref reference" data-mw='{"name":"ref","attrs":{"name":"booster","group":"lower-alpha"},"body":{"id":"mw-reference-text-cite_note-booster-11"},"parts":[{"template":{"target":{"wt":"efn","href":"./Template:Efn"},"params":{"name":{"wt":"booster"},"1":{"wt":"Falcon 9 first-stage boosters are designated with 

You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
column_names = []
for th in first_launch_table.find_all('th', scope='col'):
    name = extract_column_from_header(BeautifulSoup(str(th), 'html.parser').th)
    if name and name not in column_names:
        column_names.append(name)
print('Columns:', column_names)

Columns: ['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
fields = ['Flight No.', 'Date', 'Time', 'Version Booster', 'Launch site',
          'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']
launch_dict = {key: [] for key in fields}

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
# Parse flight-number rows in the historical Wikipedia launch tables.
# Use get_text rather than assuming each cell contains an <a> tag.
import re

def cell_text(cell):
    return cell.get_text(' ', strip=True) if cell else None

def first_text(cell):
    if cell is None:
        return None
    link = cell.find('a')
    return cell_text(link) if link and cell_text(link) else cell_text(cell)

extracted_row = 0
for table in launch_tables:
    for tr in table.find_all('tr'):
        heading = tr.find('th')
        if heading is None:
            continue
        flight_match = re.match(r'^\s*(\d+)\s*$', heading.get_text(' ', strip=True))
        if not flight_match:
            continue
        row = tr.find_all('td', recursive=False)
        if len(row) < 9:
            continue
        dt = date_time(row[0])
        date = dt[0].strip(',') if len(dt) > 0 else None
        time = dt[1] if len(dt) > 1 else None
        bv = booster_version(row[1]).strip() or first_text(row[1])
        record = {
            'Flight No.': int(flight_match.group(1)),
            'Date': date,
            'Time': time,
            'Version Booster': bv,
            'Launch site': first_text(row[2]),
            'Payload': first_text(row[3]),
            'Payload mass': get_mass(row[4]),
            'Orbit': first_text(row[5]),
            'Customer': first_text(row[6]),
            'Launch outcome': cell_text(row[7]),
            'Booster landing': landing_status(row[8]).strip() if cell_text(row[8]) else None
        }
        for key in fields:
            launch_dict[key].append(record[key])
        extracted_row += 1
print('Launch rows extracted:', extracted_row)
if extracted_row == 0:
    raise RuntimeError('No launch rows extracted; inspect the Wikipedia table structure.')

Launch rows extracted: 121


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
df = pd.DataFrame(launch_dict)
print('Shape:', df.shape)
display(df.head())
assert len(df) == extracted_row
assert df['Flight No.'].notna().all()
df.to_csv('spacex_web_scraped.csv', index=False)
print('Saved spacex_web_scraped.csv')

Shape: (121, 11)


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.07B0003.1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure
1,2,8 December 2010,15:43,F9 v1.07B0004.1,CCAFS,Dragon,0,LEO,NASA,Success [ 9 ],Failure
2,3,22 May 2012,07:44,F9 v1.07B0005.1,CCAFS,Dragon,525 kg,LEO,NASA,Success [ 20 ],No
3,4,8 October 2012,00:35,F9 v1.07B0006.1,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,No attempt
4,5,1 March 2013,15:10,F9 v1.07B0007.1,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,No


Saved spacex_web_scraped.csv


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
